In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load variables from .env file
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key

model = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=groq_api_key)

# Middleware
- Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following.

##### Tracking agent behavior with logging analysis and debugging ###
##### Transformation prompts, tool selection and output formating ###
##### Adding retries, fallback and early termination logic ###
##### Applying rate limits, guardrails and PII detection ###

## Summarization Middleware
automatically summarize conversation history when approaching token limits, preserving recent message while compressing older context summarization is used for the following:
- Long-running conversation that exceed context windows.
- Multi-turn dialogues with extensive history.
- Application where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

## agent 
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [ ]:
# Run with thread id 
config ={"configurable":{"thread_id":"test-1"}}

In [ ]:
questions = [
    "what is 2+2",
    "what is blue whale",
    "what is 8*8",
    "what is 4*4"
]
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

questions = [
    "what is 2+2",
    "what is blue whale",
    "what is 8*8",
    "what is 4*4"
]
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

## Token size

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# Load variables from .env file
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens """
    return f"""Hotel in {city}:
    1. Grand hotel - 5 star, $350/night, spa, pool, gym
    2. City inn - 4 star, $180/night, business center
    3. Budget stay - 3 star, $75/night, free wifi """

# 1. Instantiate the model object using ChatGroq
model = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=groq_api_key)

# 2. Token count function
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

# 3. Create the agent, passing the model and the custom token_counter
agent = create_agent(
    model=model,
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 550),
            keep=("tokens", 200),
            token_counter=count_tokens  # pass custom token counter here
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New york"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotel in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])}")
    print(f"{(response['messages'])}")